# 01 Load Excel Files

* Author: Jeremiah Hansen
* Last Updated: 2/27/2026

This notebook will load data into the `LOCATION` and `ORDER_DETAIL` tables from Excel files.

This currently does not use Snowpark File Access as it doesn't yet work in Notebooks. So for now we copy the file locally first.

In [ ]:
# Import python packages
import sys
import logging

# Set up the logger
logger_name = 'demo_logger'
logger = logging.getLogger(logger_name)
logger.setLevel(logging.INFO)

# Set default values for debugging
notebook_name = '01_load_excel_files.ipynb'
database_name = 'DEMO_DB'
schema_name = 'DEV_SCHEMA'
role_name = 'DEMO_ROLE'

# Override values with passed notebook arguments
if sys.argv[0].endswith('.ipynb'):
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--database-name', type=str)
    parser.add_argument('--schema-name', type=str)
    parser.add_argument('--role-name', type=str)
    args, args_unknown = parser.parse_known_args()

    notebook_name = parser.prog  # same as argv[0]
    database_name = args.database_name or database_name
    schema_name = args.schema_name or schema_name
    role_name = args.role_name or role_name

# Get a Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set the role first to avoid stale role errors
session.use_role(role_name)

# Set the default database and schema for the following cells
session.use_schema(f"{database_name}.{schema_name}")

# Get details about the current state
current_state_df = session.sql(f"""
        SELECT OBJECT_CONSTRUCT(
            'current_user', CURRENT_USER(),
            'current_role', CURRENT_ROLE(),
            'current_secondary_roles', PARSE_JSON(CURRENT_SECONDARY_ROLES()),
            'current_database', CURRENT_DATABASE(),
            'current_schema', CURRENT_SCHEMA()
        )::STRING AS session_context;
    """).collect()

logger.info(f"Begin executing notebook {notebook_name}", extra = {'logger_name': logger_name})
logger.info(f"Using parameters database: {database_name}, schema: {schema_name}, role: {role_name}", extra = {'logger_name': logger_name})
logger.info(f"Using session context {current_state_df[0]['SESSION_CONTEXT']}", extra = {'logger_name': logger_name})

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
print('Using built-in zipfile + xml to parse xlsx (no openpyxl needed)')

In [ ]:
%%sql -r dataframe_1
-- Temporary solution to load in the metadata, this should be replaced with a directy query to a directory table (or a metadata table)
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/order_detail.xlsx' AS STAGE_FILE_PATH, 'order_detail' AS WORKSHEET_NAME, 'ORDER_DETAIL' AS TARGET_TABLE
UNION
SELECT '@INTEGRATIONS.FROSTBYTE_RAW_STAGE/intro/location.xlsx', 'location', 'LOCATION';

## Create a function to load Excel worksheet to table

Create a reusable function to load an Excel worksheet to a table in Snowflake.

Note: Until we can use scoped URLs in Notebooks, via the `BUILD_SCOPED_FILE_URL()` function, we need to temporarily copy the file to a temp stage and then process from there.

In [ ]:
from snowflake.snowpark.files import SnowflakeFile
import zipfile
import xml.etree.ElementTree as ET
from io import BytesIO
import re
import pandas as pd

session.sql("CREATE TEMP STAGE IF NOT EXISTS temp_excel_stage").collect()

NS = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
      'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'}

def _col_to_idx(col_str):
    result = 0
    for c in col_str:
        result = result * 26 + (ord(c) - ord('A') + 1)
    return result - 1

def _parse_xlsx(file_bytes, worksheet_name):
    """Parse an xlsx file using zipfile + xml (no openpyxl needed)."""
    zf = zipfile.ZipFile(BytesIO(file_bytes))

    shared_strings = []
    if 'xl/sharedStrings.xml' in zf.namelist():
        ss_tree = ET.parse(zf.open('xl/sharedStrings.xml'))
        for si in ss_tree.findall('.//s:si', NS):
            texts = si.findall('.//s:t', NS)
            shared_strings.append(''.join(t.text or '' for t in texts))

    wb_tree = ET.parse(zf.open('xl/workbook.xml'))
    sheet_id = None
    for sheet in wb_tree.findall('.//s:sheets/s:sheet', NS):
        if sheet.get('name') == worksheet_name:
            sheet_id = sheet.get('sheetId')
            r_id = sheet.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
            break
    if sheet_id is None:
        raise ValueError(f"Worksheet '{worksheet_name}' not found")

    rels_tree = ET.parse(zf.open('xl/_rels/workbook.xml.rels'))
    sheet_file = None
    for rel in rels_tree.findall('.//{http://schemas.openxmlformats.org/package/2006/relationships}Relationship'):
        if rel.get('Id') == r_id:
            sheet_file = 'xl/' + rel.get('Target')
            break

    sheet_tree = ET.parse(zf.open(sheet_file))

    max_col = 0
    raw_rows = []
    for row in sheet_tree.findall('.//s:sheetData/s:row', NS):
        cells = []
        for cell in row.findall('s:c', NS):
            ref = cell.get('r')
            col_str = re.match(r'[A-Z]+', ref).group()
            col_idx = _col_to_idx(col_str)
            val_el = cell.find('s:v', NS)
            if val_el is None or val_el.text is None:
                val = None
            elif cell.get('t') == 's':
                val = shared_strings[int(val_el.text)]
            else:
                try:
                    num = float(val_el.text)
                    val = int(num) if num == int(num) else num
                except ValueError:
                    val = val_el.text
            cells.append((col_idx, val))
            if col_idx + 1 > max_col:
                max_col = col_idx + 1
        raw_rows.append(cells)

    rows_data = []
    for cells in raw_rows:
        row_vals = [None] * max_col
        for col_idx, val in cells:
            row_vals[col_idx] = val
        rows_data.append(row_vals)

    columns = rows_data[0]
    data = rows_data[1:]
    return pd.DataFrame(data, columns=columns)

def load_excel_worksheet_to_table(session, external_path, worksheet_name, target_table):
    """Load an Excel worksheet by copying to internal stage first"""
    filename = external_path.split('/')[-1]

    session.sql(f"""
        COPY FILES INTO @temp_excel_stage
        FROM {external_path}
    """).collect()

    with SnowflakeFile.open(f'@temp_excel_stage/{filename}', 'rb') as f:
        file_bytes = f.read()

    df = _parse_xlsx(file_bytes, worksheet_name)

    snowpark_df = session.create_dataframe(df)
    snowpark_df.write.mode("overwrite").save_as_table(target_table)

    logger.info(f"Loaded {len(df)} rows from '{worksheet_name}' to {target_table}", extra = {'logger_name': logger_name})

## Process all Excel worksheets

Loop through each Excel worksheet to process and call our `load_excel_worksheet_to_table_local()` function.

In [ ]:
# Process each file from the sql_get_spreadsheets cell above
files_to_load = dataframe_1
for index, excel_file in files_to_load.iterrows():
    print(f"Processing Excel file {excel_file['STAGE_FILE_PATH']}")
    load_excel_worksheet_to_table(session, excel_file['STAGE_FILE_PATH'], excel_file['WORKSHEET_NAME'], excel_file['TARGET_TABLE'])

logger.info(f"Finish executing notebook {notebook_name}", extra = {'logger_name': logger_name})

### Debugging

In [ ]:
%%sql -r dataframe_2
--DESCRIBE TABLE LOCATION;
--SELECT * FROM LOCATION;
SHOW TABLES;